# キーポイント＋人物輪郭で踊る「線画パペット」

このノートブックは、棒人間ではなく、**白い面＋黒い輪郭線**を持つ2Dキャラクターをダンサーの動きに追従させるPoCです。

- Poseモデルの17キーポイントで、対象人物・関節位置・動きを追跡します。
- Segmentationモデルの人物マスクから、外周輪郭と各部位の太さを測ります。
- マスクの距離変換で測った太さを骨格へ戻し、頭・胴体・腕・脚を一体の輪郭ボディとして再構成します。
- 全体外周とは別に頭・左右の腕・左右の脚の輪郭線を保持するため、胴体へ重なっても白い塊になりません。
- 元マスク、再構成ボディ、両者のハイブリッドを切り替えられます。
- 元動画へ並べた動画と、キャラクターだけの動画を出力します。

> これは「そのフレームのダンサー輪郭を線画化する」方式です。固定イラスト1枚を別の動きへ変形する完全な2Dリグではありません。髪・服・顔を同じデザインに固定したい場合は、部位分けPSD/PNGまたはメッシュリグへ次段階で移行します。

Colabへ動画をアップロードする場合は外部転送になります。権利・プライバシーを確認し、次の入力セルで明示的に許可してください。Ultralyticsモデルの利用条件も用途に合わせて確認してください。

## 1. 依存関係

ColabのGPUランタイムを推奨します。インストール後のランタイム再起動は不要です。

In [ ]:
import subprocess
import sys

subprocess.run(['nvidia-smi'], check=False)
subprocess.check_call([
    sys.executable,
    '-m',
    'pip',
    'install',
    '-q',
    '-U',
    'ultralytics>=8.3,<9',
    'opencv-python-headless>=4.10,<5',
    'tqdm>=4.66,<5',
])
print('Dependencies are ready.')

## 2. 入力動画

`ALLOW_EXTERNAL_VIDEO_TRANSFER = True` は、動画をGoogle Colabランタイムへ転送してよい場合だけ設定してください。Driveを使う場合も転送に該当します。

In [ ]:
import hashlib
import shutil
from pathlib import Path

ALLOW_EXTERNAL_VIDEO_TRANSFER = False  # 権利・プライバシーを確認後、実行時に True へ変更
USE_GOOGLE_DRIVE = False
DRIVE_VIDEO_PATH = '/content/drive/MyDrive/dance_input.mp4'

if not ALLOW_EXTERNAL_VIDEO_TRANSFER:
    raise RuntimeError(
        '動画のColab転送は未許可です。権利・プライバシーを確認し、'
        'ALLOW_EXTERNAL_VIDEO_TRANSFER=True にしてこのセルを再実行してください。'
    )

if USE_GOOGLE_DRIVE:
    from google.colab import drive

    drive.mount('/content/drive', force_remount=False)
    source_video = Path(DRIVE_VIDEO_PATH)
    if not source_video.is_file():
        raise FileNotFoundError(source_video)
    video_path = Path('/content') / source_video.name
    shutil.copy2(source_video, video_path)
else:
    from google.colab import files

    uploaded = files.upload()
    video_names = [name for name in uploaded if Path(name).suffix.lower() in {'.mp4', '.mov', '.mkv', '.avi'}]
    if len(video_names) != 1:
        raise ValueError(f'動画を1本だけ選択してください: {video_names}')
    video_path = Path('/content') / Path(video_names[0]).name
    video_path.write_bytes(uploaded[video_names[0]])
    del uploaded

def _sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


input_sha256 = _sha256_file(video_path)
print({'video': video_path.name, 'bytes': video_path.stat().st_size, 'sha256': input_sha256})

## 3. モデルと描画設定

`PUPPET_MODE` は次の3種類です。

- `'capsule'`: キーポイント＋マスクから測った部位幅で、滑らかな輪郭ボディを再構成（最もキャラクター的）
- `'silhouette'`: ダンサーの現在フレームの輪郭をそのまま線画化（最も忠実）
- `'hybrid'`: 上記を合成（既定、欠けに強い）

どのモードでも `PRESERVE_PART_OUTLINES=True` の場合、全体外周とは独立した頭・左右腕・左右脚の線を最後に上描きします。腕が胸や腹へ重なっても部位線は消えません。

複数人動画で別人を選ぶ場合は、一度自動実行して表示されたIDを `TARGET_TRACK_ID` に設定して再実行します。

In [ ]:
import json
import math
from dataclasses import dataclass

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
from tqdm.auto import tqdm
from ultralytics import YOLO

OUTPUT_DIR = Path('/content/contour_puppet_outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Inference
POSE_MODEL_NAME = 'yolov8m-pose.pt'  # 高精度寄り: yolov8x-pose.pt / 高速: yolov8n-pose.pt
SEG_MODEL_NAME = 'yolov8m-seg.pt'    # 高精度寄り: yolov8x-seg.pt / 高速: yolov8n-seg.pt
POSE_CONFIDENCE = 0.20
SEG_CONFIDENCE = 0.25
KEYPOINT_CONFIDENCE = 0.20
INFERENCE_IMAGE_SIZE = 960
TARGET_TRACK_ID = None
FRAME_STRIDE = 1
MAX_OUTPUT_FRAMES = None  # 動作確認例: 180
MAX_HOLD_FRAMES = 8
POSE_SMOOTHING_ALPHA = 0.48
BBOX_SMOOTHING_ALPHA = 0.32

# Character geometry
PUPPET_MODE = 'capsule'  # 'capsule', 'silhouette', 'hybrid'
PUPPET_STYLE = 'slim_cute'  # 'slim_cute', 'source_proportions'
PUPPET_SCALE = 0.72
PUPPET_SIDE = 'auto'    # 'auto', 'left', 'right'
PUPPET_GAP_IN_HEIGHT = 0.12
SLIM_ARM_WIDTH_RATIO = 0.042
SLIM_LEG_WIDTH_RATIO = 0.060
SLIM_SHOULDER_HALF_RATIO = 0.105
SLIM_HIP_HALF_RATIO = 0.080
CUTE_HEAD_RADIUS_X_RATIO = 0.165
CUTE_HEAD_RADIUS_Y_RATIO = 0.145
MASK_THRESHOLD = 0.45
MASK_CLOSE_RATIO = 0.008

# Line-art style (OpenCV uses BGR)
FILL_BGR = (250, 250, 245)
OUTLINE_BGR = (20, 20, 20)
STAGE_BGR = (236, 240, 244)
OUTLINE_WIDTH = 3
DRAW_FACE = True
DRAW_EARS = True
PRESERVE_PART_OUTLINES = True  # 重なっても頭・腕・脚の線を最前面に残す
PART_OUTLINE_WIDTH = 2
DRAW_INNER_GESTURE_LINES = False
OVERLAY_OPACITY = 0.97
PREVIEW_FRAME_INDEX = 0

if PUPPET_MODE not in {'capsule', 'silhouette', 'hybrid'}:
    raise ValueError(PUPPET_MODE)
if PUPPET_STYLE not in {'slim_cute', 'source_proportions'}:
    raise ValueError(PUPPET_STYLE)
if PUPPET_SIDE not in {'auto', 'left', 'right'}:
    raise ValueError(PUPPET_SIDE)
if FRAME_STRIDE < 1 or PUPPET_SCALE <= 0:
    raise ValueError('FRAME_STRIDE and PUPPET_SCALE must be positive.')

device_arg = 0 if torch.cuda.is_available() else 'cpu'
pose_model = YOLO(POSE_MODEL_NAME)
seg_model = YOLO(SEG_MODEL_NAME)
print({'device': device_arg, 'pose_model': POSE_MODEL_NAME, 'seg_model': SEG_MODEL_NAME})
print('License note: confirm the Ultralytics model/license terms for the intended deployment.')

## 4. 輪郭パペット生成関数

人物マスクの距離変換値は、その画素から輪郭までの距離です。骨の途中でこの値を測ると腕・脚・胴体の局所的な半幅になり、棒ではなく「厚みのある輪郭人形」を作れます。

In [ ]:
COCO = {
    'nose': 0,
    'left_eye': 1,
    'right_eye': 2,
    'left_ear': 3,
    'right_ear': 4,
    'left_shoulder': 5,
    'right_shoulder': 6,
    'left_elbow': 7,
    'right_elbow': 8,
    'left_wrist': 9,
    'right_wrist': 10,
    'left_hip': 11,
    'right_hip': 12,
    'left_knee': 13,
    'right_knee': 14,
    'left_ankle': 15,
    'right_ankle': 16,
}

LIMB_CHAINS = [
    ((5, 7, 9), ((0.045, 0.105), (0.035, 0.085))),
    ((6, 8, 10), ((0.045, 0.105), (0.035, 0.085))),
    ((11, 13, 15), ((0.070, 0.145), (0.050, 0.115))),
    ((12, 14, 16), ((0.070, 0.145), (0.050, 0.115))),
]


@dataclass
class TrackState:
    track_id: int | None = None
    pose_xy: np.ndarray | None = None
    pose_conf: np.ndarray | None = None
    bbox: np.ndarray | None = None
    mask: np.ndarray | None = None
    missing_frames: int = 0
    placement_side: str | None = None


def _as_numpy(value, empty_shape, dtype=np.float32):
    if value is None:
        return np.empty(empty_shape, dtype=dtype)
    if hasattr(value, 'detach'):
        value = value.detach().cpu().numpy()
    array = np.asarray(value)
    if array.size == 0:
        return np.empty(empty_shape, dtype=dtype)
    return array.astype(dtype, copy=False)


def _bbox_iou(a, b):
    if a is None or b is None:
        return 0.0
    x1 = max(float(a[0]), float(b[0]))
    y1 = max(float(a[1]), float(b[1]))
    x2 = min(float(a[2]), float(b[2]))
    y2 = min(float(a[3]), float(b[3]))
    inter = max(0.0, x2 - x1) * max(0.0, y2 - y1)
    area_a = max(0.0, float(a[2] - a[0])) * max(0.0, float(a[3] - a[1]))
    area_b = max(0.0, float(b[2] - b[0])) * max(0.0, float(b[3] - b[1]))
    return inter / max(area_a + area_b - inter, 1e-6)


def _pose_arrays(result):
    keypoints = getattr(result, 'keypoints', None)
    boxes_obj = getattr(result, 'boxes', None)
    xy = _as_numpy(getattr(keypoints, 'xy', None), (0, 17, 2))
    conf = _as_numpy(getattr(keypoints, 'conf', None), (len(xy), 17))
    if len(xy) and conf.size == 0:
        conf = np.ones((len(xy), 17), dtype=np.float32)
    boxes = _as_numpy(getattr(boxes_obj, 'xyxy', None), (len(xy), 4))
    det_conf = _as_numpy(getattr(boxes_obj, 'conf', None), (len(xy),))
    raw_ids = _as_numpy(getattr(boxes_obj, 'id', None), (0,), dtype=np.float32)
    track_ids = [int(raw_ids[i]) if i < len(raw_ids) and np.isfinite(raw_ids[i]) else None for i in range(len(xy))]
    return xy, conf, boxes, det_conf, track_ids


def _select_pose_person(result, state):
    xy, conf, boxes, det_conf, track_ids = _pose_arrays(result)
    if len(xy) == 0:
        return None

    preferred_id = TARGET_TRACK_ID if TARGET_TRACK_ID is not None else state.track_id
    if preferred_id is not None:
        for index, track_id in enumerate(track_ids):
            if track_id == int(preferred_id):
                return index, xy, conf, boxes, det_conf, track_ids
        if TARGET_TRACK_ID is not None or state.track_id is not None:
            return None

    frame_area = max(1.0, float(result.orig_shape[0] * result.orig_shape[1]))
    scores = []
    for index, bbox in enumerate(boxes):
        area = max(0.0, float((bbox[2] - bbox[0]) * (bbox[3] - bbox[1]))) / frame_area
        continuity = _bbox_iou(bbox, state.bbox)
        confidence = float(det_conf[index]) if index < len(det_conf) else 0.0
        scores.append(2.5 * continuity + 0.8 * area + 0.4 * confidence)
    index = int(np.argmax(scores))
    return index, xy, conf, boxes, det_conf, track_ids


def _canonical_pose_from_bbox(bbox):
    x1, y1, x2, y2 = [float(value) for value in bbox]
    width = max(8.0, x2 - x1)
    height = max(8.0, y2 - y1)
    normalized = np.asarray([
        [0.50, 0.10], [0.46, 0.085], [0.54, 0.085], [0.40, 0.11], [0.60, 0.11],
        [0.38, 0.25], [0.62, 0.25], [0.31, 0.42], [0.69, 0.42], [0.27, 0.58],
        [0.73, 0.58], [0.43, 0.55], [0.57, 0.55], [0.42, 0.75], [0.58, 0.75],
        [0.40, 0.96], [0.60, 0.96],
    ], dtype=np.float32)
    completed = normalized * np.asarray([width, height], dtype=np.float32)
    completed += np.asarray([x1, y1], dtype=np.float32)
    return completed


def _update_pose_state(state, selected):
    if selected is None:
        state.missing_frames += 1
        return False

    index, xy, conf, boxes, _, track_ids = selected
    current_xy = xy[index].astype(np.float32)
    current_conf = conf[index].astype(np.float32)
    current_bbox = boxes[index].astype(np.float32)
    valid = np.isfinite(current_xy).all(axis=1) & (current_conf >= KEYPOINT_CONFIDENCE)

    if state.pose_xy is None:
        smoothed = _canonical_pose_from_bbox(current_bbox)
        smoothed[valid] = current_xy[valid]
    else:
        smoothed = state.pose_xy.copy()
        smoothed[valid] = (
            POSE_SMOOTHING_ALPHA * current_xy[valid]
            + (1.0 - POSE_SMOOTHING_ALPHA) * state.pose_xy[valid]
        )

    if state.bbox is None:
        state.bbox = current_bbox
    else:
        state.bbox = (
            BBOX_SMOOTHING_ALPHA * current_bbox
            + (1.0 - BBOX_SMOOTHING_ALPHA) * state.bbox
        )
    if state.pose_conf is None:
        effective_conf = current_conf.copy()
    else:
        effective_conf = state.pose_conf.copy()
        effective_conf[valid] = current_conf[valid]
    effective_conf = np.nan_to_num(effective_conf, nan=0.0, posinf=0.0, neginf=0.0)
    effective_conf[~valid] = np.maximum(effective_conf[~valid], KEYPOINT_CONFIDENCE * 0.50)
    state.pose_xy = smoothed
    state.pose_conf = effective_conf
    if track_ids[index] is not None:
        state.track_id = track_ids[index]
    state.missing_frames = 0
    return True


def _clean_mask(mask_float, frame_shape):
    frame_h, frame_w = frame_shape[:2]
    if mask_float.shape[:2] != (frame_h, frame_w):
        mask_float = cv2.resize(mask_float, (frame_w, frame_h), interpolation=cv2.INTER_LINEAR)
    mask = (mask_float >= MASK_THRESHOLD).astype(np.uint8) * 255
    kernel_size = max(3, int(round(min(frame_h, frame_w) * MASK_CLOSE_RATIO)))
    if kernel_size % 2 == 0:
        kernel_size += 1
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (kernel_size, kernel_size))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    count, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    if count <= 1:
        return None
    largest = 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))
    return np.where(labels == largest, 255, 0).astype(np.uint8)


def _select_person_mask(seg_result, pose_bbox, frame_shape):
    boxes_obj = getattr(seg_result, 'boxes', None)
    masks_obj = getattr(seg_result, 'masks', None)
    if boxes_obj is None or masks_obj is None or getattr(masks_obj, 'data', None) is None:
        return None
    boxes = _as_numpy(getattr(boxes_obj, 'xyxy', None), (0, 4))
    classes = _as_numpy(getattr(boxes_obj, 'cls', None), (len(boxes),))
    masks = _as_numpy(masks_obj.data, (0, 1, 1))
    candidates = [i for i in range(min(len(boxes), len(masks))) if i >= len(classes) or int(classes[i]) == 0]
    if not candidates:
        return None
    best_index = max(candidates, key=lambda i: _bbox_iou(boxes[i], pose_bbox))
    if _bbox_iou(boxes[best_index], pose_bbox) < 0.05:
        return None
    return _clean_mask(masks[best_index], frame_shape)


def _infer_person(frame_bgr, state, persist_tracking=True):
    pose_result = pose_model.track(
        frame_bgr,
        conf=POSE_CONFIDENCE,
        imgsz=INFERENCE_IMAGE_SIZE,
        device=device_arg,
        persist=persist_tracking,
        tracker='bytetrack.yaml',
        verbose=False,
    )[0]
    selected = _select_pose_person(pose_result, state)
    pose_updated = _update_pose_state(state, selected)

    if pose_updated:
        seg_result = seg_model.predict(
            frame_bgr,
            conf=SEG_CONFIDENCE,
            imgsz=INFERENCE_IMAGE_SIZE,
            device=device_arg,
            retina_masks=True,
            verbose=False,
        )[0]
        selected_mask = _select_person_mask(seg_result, state.bbox, frame_bgr.shape)
        if selected_mask is not None:
            state.mask = selected_mask

    usable = (
        state.pose_xy is not None
        and state.pose_conf is not None
        and state.bbox is not None
        and state.mask is not None
        and state.missing_frames <= MAX_HOLD_FRAMES
    )
    return usable, pose_result


def _resolve_overlay_side(bbox, frame_shape, current_side=None):
    if PUPPET_SIDE in {'left', 'right'}:
        return PUPPET_SIDE
    if current_side in {'left', 'right'}:
        return current_side
    _, frame_w = frame_shape[:2]
    x1, _, x2, _ = [float(value) for value in bbox]
    source_w = max(8.0, x2 - x1)
    target_w = source_w * PUPPET_SCALE
    gap = PUPPET_GAP_IN_HEIGHT * max(8.0, float(bbox[3] - bbox[1]))
    left_center = x1 - gap - 0.5 * target_w
    right_center = x2 + gap + 0.5 * target_w
    left_overflow = max(0.0, 0.5 * target_w - left_center)
    right_overflow = max(0.0, right_center + 0.5 * target_w - frame_w)
    return 'left' if left_overflow <= right_overflow else 'right'


def _placement_matrix(bbox, frame_shape, placement, overlay_side=None):
    frame_h, frame_w = frame_shape[:2]
    x1, y1, x2, y2 = [float(value) for value in bbox]
    source_w = max(8.0, x2 - x1)
    source_h = max(8.0, y2 - y1)
    source_center_x = 0.5 * (x1 + x2)

    if placement == 'stage':
        scale = min(1.20, 0.82 * frame_h / source_h)
        target_center_x = 0.5 * frame_w
        target_bottom = 0.94 * frame_h
    else:
        scale = PUPPET_SCALE
        target_w = source_w * scale
        gap = PUPPET_GAP_IN_HEIGHT * source_h
        left_center = x1 - gap - 0.5 * target_w
        right_center = x2 + gap + 0.5 * target_w
        resolved_side = _resolve_overlay_side(bbox, frame_shape, overlay_side)
        if resolved_side == 'left':
            target_center_x = left_center
        else:
            target_center_x = right_center
        target_center_x = float(np.clip(target_center_x, 0.5 * target_w + 2, frame_w - 0.5 * target_w - 2))
        target_bottom = min(frame_h - 3.0, y2)

    matrix = np.asarray([
        [scale, 0.0, target_center_x - scale * source_center_x],
        [0.0, scale, target_bottom - scale * y2],
    ], dtype=np.float32)
    return matrix, scale, source_h, target_bottom


def _transform_points(points, matrix):
    homogeneous = np.concatenate([points.astype(np.float32), np.ones((len(points), 1), dtype=np.float32)], axis=1)
    return homogeneous @ matrix.T


def _distance_near(distance_map, point, patch_radius):
    frame_h, frame_w = distance_map.shape
    x = int(round(float(point[0])))
    y = int(round(float(point[1])))
    radius = max(1, int(patch_radius))
    x1, x2 = max(0, x - radius), min(frame_w, x + radius + 1)
    y1, y2 = max(0, y - radius), min(frame_h, y + radius + 1)
    if x1 >= x2 or y1 >= y2:
        return 0.0
    values = distance_map[y1:y2, x1:x2]
    return float(np.percentile(values, 80)) if values.size else 0.0


def _bone_width(distance_map, point_a, point_b, source_height, low_ratio, high_ratio):
    samples = []
    for amount in (0.25, 0.50, 0.75):
        point = (1.0 - amount) * point_a + amount * point_b
        samples.append(2.0 * _distance_near(distance_map, point, 0.012 * source_height))
    width = float(np.median(samples))
    return float(np.clip(width, low_ratio * source_height, high_ratio * source_height))


def _component_outline(component_mask, open_joint=None, open_radius=0):
    component_lines = np.zeros_like(component_mask)
    contours, _ = cv2.findContours(component_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_TC89_KCOS)
    cv2.drawContours(component_lines, contours, -1, 255, PART_OUTLINE_WIDTH, cv2.LINE_AA)
    if open_joint is not None and open_radius > 0:
        cv2.circle(
            component_lines,
            tuple(np.rint(open_joint).astype(int)),
            int(open_radius),
            0,
            -1,
            cv2.LINE_AA,
        )
    return component_lines


def _capsule_body_mask(source_mask, source_pose, target_pose, pose_conf, scale, source_height):
    frame_h, frame_w = source_mask.shape
    body = np.zeros((frame_h, frame_w), dtype=np.uint8)
    part_line_mask = np.zeros((frame_h, frame_w), dtype=np.uint8)
    distance_map = cv2.distanceTransform((source_mask > 0).astype(np.uint8), cv2.DIST_L2, 5)

    for joints, width_ranges in LIMB_CHAINS:
        limb_mask = np.zeros_like(body)
        proximal_thickness = 3
        for segment_index, (joint_a, joint_b) in enumerate(zip(joints[:-1], joints[1:])):
            if PUPPET_STYLE == 'slim_cute':
                width_ratio = SLIM_ARM_WIDTH_RATIO if joints[0] in {5, 6} else SLIM_LEG_WIDTH_RATIO
                taper = 1.08 if segment_index == 0 else 0.88
                source_width = width_ratio * source_height * taper
            else:
                low_ratio, high_ratio = width_ranges[segment_index]
                source_width = _bone_width(
                    distance_map,
                    source_pose[joint_a],
                    source_pose[joint_b],
                    source_height,
                    low_ratio,
                    high_ratio,
                )
            thickness = max(3, int(round(source_width * scale)))
            if segment_index == 0:
                proximal_thickness = thickness
            point_a = tuple(np.rint(target_pose[joint_a]).astype(int))
            point_b = tuple(np.rint(target_pose[joint_b]).astype(int))
            cv2.line(limb_mask, point_a, point_b, 255, thickness, cv2.LINE_AA)
        body = cv2.bitwise_or(body, limb_mask)
        if PUPPET_STYLE != 'slim_cute' or joints[0] in {5, 6}:
            limb_lines = _component_outline(
                limb_mask,
                open_joint=target_pose[joints[0]],
                open_radius=max(2, int(round(0.60 * proximal_thickness))),
            )
            part_line_mask = cv2.bitwise_or(part_line_mask, limb_lines)

    shoulder_source = 0.5 * (source_pose[5] + source_pose[6])
    hip_source = 0.5 * (source_pose[11] + source_pose[12])
    shoulder_target = 0.5 * (target_pose[5] + target_pose[6])
    hip_target = 0.5 * (target_pose[11] + target_pose[12])
    axis = hip_target - shoulder_target
    axis_norm = float(np.linalg.norm(axis))
    perpendicular = np.asarray([1.0, 0.0], dtype=np.float32) if axis_norm < 1e-4 else np.asarray([-axis[1], axis[0]]) / axis_norm
    shoulder_half_source = max(
        _distance_near(distance_map, shoulder_source, 0.02 * source_height),
        0.5 * float(np.linalg.norm(source_pose[5] - source_pose[6])),
    )
    hip_half_source = max(
        _distance_near(distance_map, hip_source, 0.02 * source_height),
        0.5 * float(np.linalg.norm(source_pose[11] - source_pose[12])),
    )
    if PUPPET_STYLE == 'slim_cute':
        shoulder_half = SLIM_SHOULDER_HALF_RATIO * source_height * scale
        hip_half = SLIM_HIP_HALF_RATIO * source_height * scale
    else:
        shoulder_half = np.clip(shoulder_half_source, 0.10 * source_height, 0.24 * source_height) * scale
        hip_half = np.clip(hip_half_source, 0.08 * source_height, 0.20 * source_height) * scale
    torso = np.asarray([
        shoulder_target - perpendicular * shoulder_half,
        shoulder_target + perpendicular * shoulder_half,
        hip_target + perpendicular * hip_half,
        hip_target - perpendicular * hip_half,
    ], dtype=np.int32)
    torso_mask = np.zeros_like(body)
    cv2.fillConvexPoly(torso_mask, torso, 255, cv2.LINE_AA)
    body = cv2.bitwise_or(body, torso_mask)

    nose_source = source_pose[COCO['nose']]
    nose_target = target_pose[COCO['nose']]
    head_radius_source = _distance_near(distance_map, nose_source, 0.025 * source_height)
    if PUPPET_STYLE == 'slim_cute':
        head_radius_x = int(round(CUTE_HEAD_RADIUS_X_RATIO * source_height * scale))
        head_radius_y = int(round(CUTE_HEAD_RADIUS_Y_RATIO * source_height * scale))
    else:
        head_radius_x = int(round(np.clip(head_radius_source, 0.075 * source_height, 0.145 * source_height) * scale))
        head_radius_y = int(round(np.clip(head_radius_source * 1.08, 0.085 * source_height, 0.165 * source_height) * scale))
    head_center = tuple(np.rint(nose_target + np.asarray([0.0, -0.015 * source_height * scale])).astype(int))
    head_mask = np.zeros_like(body)
    if PUPPET_STYLE == 'slim_cute' and DRAW_EARS:
        ear_radius = (max(3, int(round(0.34 * head_radius_x))), max(3, int(round(0.28 * head_radius_y))))
        ear_offset_x = int(round(0.72 * head_radius_x))
        ear_offset_y = int(round(0.72 * head_radius_y))
        for direction in (-1, 1):
            ear_center = (head_center[0] + direction * ear_offset_x, head_center[1] - ear_offset_y)
            cv2.ellipse(head_mask, ear_center, ear_radius, direction * 18, 0, 360, 255, -1, cv2.LINE_AA)
    cv2.ellipse(head_mask, head_center, (max(3, head_radius_x), max(3, head_radius_y)), 0, 0, 360, 255, -1, cv2.LINE_AA)
    body = cv2.bitwise_or(body, head_mask)
    part_line_mask = cv2.bitwise_or(part_line_mask, _component_outline(head_mask))
    neck_ratio = 0.045 if PUPPET_STYLE == 'slim_cute' else 0.07
    neck_thickness = max(3, int(round(neck_ratio * source_height * scale)))
    cv2.line(body, head_center, tuple(np.rint(shoulder_target).astype(int)), 255, neck_thickness, cv2.LINE_AA)

    smooth_size = max(3, int(round(0.012 * source_height * scale)))
    if smooth_size % 2 == 0:
        smooth_size += 1
    body = cv2.GaussianBlur(body, (smooth_size, smooth_size), 0)
    body = np.where(body >= 96, 255, 0).astype(np.uint8)
    return body, part_line_mask


def _styled_layer(puppet_mask, part_line_mask, target_pose, pose_conf, target_height):
    frame_h, frame_w = puppet_mask.shape
    layer = np.zeros((frame_h, frame_w, 3), dtype=np.uint8)
    alpha = np.zeros((frame_h, frame_w), dtype=np.uint8)
    contours, _ = cv2.findContours(puppet_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_TC89_KCOS)
    if not contours:
        return layer, alpha

    layer[puppet_mask > 0] = FILL_BGR
    alpha[puppet_mask > 0] = int(round(255 * OVERLAY_OPACITY))
    stroke_mask = np.zeros_like(puppet_mask)
    cv2.drawContours(stroke_mask, contours, -1, 255, OUTLINE_WIDTH, cv2.LINE_AA)
    layer[stroke_mask > 0] = OUTLINE_BGR
    alpha[stroke_mask > 0] = 255

    if PRESERVE_PART_OUTLINES:
        support_kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
        supported_body = cv2.dilate(puppet_mask, support_kernel, iterations=1)
        visible_part_lines = cv2.bitwise_and(part_line_mask, supported_body)
        layer[visible_part_lines > 0] = OUTLINE_BGR
        alpha[visible_part_lines > 0] = 255

    detail_thickness = max(1, OUTLINE_WIDTH // 2)
    if DRAW_INNER_GESTURE_LINES:
        for joints in ((5, 7, 9), (6, 8, 10)):
            points = np.rint(target_pose[list(joints)]).astype(np.int32).reshape((-1, 1, 2))
            cv2.polylines(layer, [points], False, OUTLINE_BGR, detail_thickness, cv2.LINE_AA)

    if DRAW_FACE:
        nose = np.rint(target_pose[0]).astype(int)
        eye_radius = max(2, int(round(0.010 * target_height)))
        for eye_index in (1, 2):
            eye = np.rint(target_pose[eye_index]).astype(int)
            if pose_conf[eye_index] >= KEYPOINT_CONFIDENCE * 0.50:
                x, y = int(eye[0]), int(eye[1])
                if 0 <= x < frame_w and 0 <= y < frame_h and puppet_mask[y, x] > 0:
                    cv2.circle(layer, (x, y), eye_radius, OUTLINE_BGR, -1, cv2.LINE_AA)
        nose_center = (int(nose[0]), int(nose[1]))
        nose_radius = max(2, int(round(0.012 * target_height)))
        cv2.ellipse(layer, nose_center, (nose_radius, max(2, nose_radius // 2)), 0, 0, 360, OUTLINE_BGR, detail_thickness, cv2.LINE_AA)
        mouth_center = nose + np.asarray([0, int(round(0.045 * target_height))])
        mouth_width = max(3, int(round(0.025 * target_height)))
        cv2.ellipse(
            layer,
            tuple(mouth_center.astype(int)),
            (mouth_width, max(2, mouth_width // 3)),
            0,
            15,
            165,
            OUTLINE_BGR,
            detail_thickness,
            cv2.LINE_AA,
        )
    return layer, alpha


def _alpha_blend(base_bgr, layer_bgr, alpha_u8):
    alpha = (alpha_u8.astype(np.float32) / 255.0)[:, :, None]
    return np.clip(base_bgr.astype(np.float32) * (1.0 - alpha) + layer_bgr.astype(np.float32) * alpha, 0, 255).astype(np.uint8)


def _render_character(base_bgr, state, placement):
    if placement != 'stage':
        state.placement_side = _resolve_overlay_side(state.bbox, base_bgr.shape, state.placement_side)
    matrix, scale, source_height, target_bottom = _placement_matrix(
        state.bbox,
        base_bgr.shape,
        placement,
        state.placement_side,
    )
    frame_h, frame_w = base_bgr.shape[:2]
    target_pose = _transform_points(state.pose_xy, matrix)
    live_mask = cv2.warpAffine(
        state.mask,
        matrix,
        (frame_w, frame_h),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=0,
    )
    live_mask = np.where(live_mask >= 96, 255, 0).astype(np.uint8)
    capsule_mask, part_line_mask = _capsule_body_mask(
        state.mask,
        state.pose_xy,
        target_pose,
        state.pose_conf,
        scale,
        source_height,
    )
    if PUPPET_MODE == 'silhouette':
        puppet_mask = live_mask
    elif PUPPET_MODE == 'capsule':
        puppet_mask = capsule_mask
    else:
        puppet_mask = cv2.bitwise_or(live_mask, capsule_mask)
        close_size = max(3, int(round(0.008 * source_height * scale)))
        if close_size % 2 == 0:
            close_size += 1
        close_kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (close_size, close_size))
        puppet_mask = cv2.morphologyEx(puppet_mask, cv2.MORPH_CLOSE, close_kernel)

    layer, alpha = _styled_layer(
        puppet_mask,
        part_line_mask,
        target_pose,
        state.pose_conf,
        source_height * scale,
    )
    rendered = _alpha_blend(base_bgr, layer, alpha)
    return rendered, puppet_mask, target_pose, target_bottom


def _diagnostic_frame(frame_bgr, state):
    diagnostic = frame_bgr.copy()
    contours, _ = cv2.findContours(state.mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(diagnostic, contours, -1, (40, 220, 120), 2, cv2.LINE_AA)
    for index, point in enumerate(state.pose_xy):
        if state.pose_conf[index] >= KEYPOINT_CONFIDENCE:
            cv2.circle(diagnostic, tuple(np.rint(point).astype(int)), 4, (30, 80, 255), -1, cv2.LINE_AA)
    x1, y1, x2, y2 = np.rint(state.bbox).astype(int)
    label = f'track={state.track_id}' if state.track_id is not None else 'track=auto'
    cv2.rectangle(diagnostic, (x1, y1), (x2, y2), (255, 180, 40), 2)
    cv2.putText(diagnostic, label, (x1, max(24, y1 - 8)), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255, 180, 40), 2, cv2.LINE_AA)
    return diagnostic

## 5. 1フレームでデザイン確認

長い動画を処理する前に、対象人物・マスク・輪郭人形を確認します。人物がまだ現れない場合は `PREVIEW_FRAME_INDEX` を変更してください。

In [ ]:
preview_capture = cv2.VideoCapture(str(video_path))
if not preview_capture.isOpened():
    raise RuntimeError(f'動画を開けません: {video_path}')
preview_capture.set(cv2.CAP_PROP_POS_FRAMES, PREVIEW_FRAME_INDEX)
preview_ok, preview_frame = preview_capture.read()
preview_capture.release()
if not preview_ok:
    raise RuntimeError(f'プレビューフレームを読めません: {PREVIEW_FRAME_INDEX}')

preview_state = TrackState()
preview_usable, _ = _infer_person(preview_frame, preview_state, persist_tracking=False)
if not preview_usable:
    raise RuntimeError('このフレームで人物姿勢または人物マスクを取得できません。PREVIEW_FRAME_INDEXを変更してください。')

preview_diagnostic = _diagnostic_frame(preview_frame, preview_state)
preview_overlay, _, _, _ = _render_character(preview_frame.copy(), preview_state, 'overlay')
preview_stage_base = np.full_like(preview_frame, STAGE_BGR)
preview_ground = int(round(0.94 * preview_stage_base.shape[0]))
cv2.line(preview_stage_base, (0, preview_ground), (preview_stage_base.shape[1] - 1, preview_ground), (180, 185, 190), 2, cv2.LINE_AA)
preview_stage, _, _, _ = _render_character(preview_stage_base, preview_state, 'stage')

figure, axes = plt.subplots(1, 3, figsize=(18, 7))
for axis, image, title in zip(
    axes,
    [preview_diagnostic, preview_overlay, preview_stage],
    ['mask + keypoints', 'source + contour puppet', 'contour puppet only'],
):
    axis.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    axis.set_title(title)
    axis.axis('off')
plt.tight_layout()
plt.show()
print({'selected_track_id': preview_state.track_id, 'mode': PUPPET_MODE, 'preview_frame': PREVIEW_FRAME_INDEX})

## 6. 動画全体をレンダリング

出力は、元動画内へ並べる `contour_puppet_overlay.mp4` と、背景上でキャラクターだけを表示する `contour_puppet_only.mp4` です。短い欠損は直前状態を最大 `MAX_HOLD_FRAMES` だけ保持します。

In [ ]:
capture = cv2.VideoCapture(str(video_path))
if not capture.isOpened():
    raise RuntimeError(f'動画を開けません: {video_path}')

source_fps = float(capture.get(cv2.CAP_PROP_FPS) or 30.0)
frame_width = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
source_frame_count = int(capture.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
output_fps = source_fps / FRAME_STRIDE

overlay_raw = OUTPUT_DIR / 'contour_puppet_overlay_raw.mp4'
puppet_raw = OUTPUT_DIR / 'contour_puppet_only_raw.mp4'
overlay_final = OUTPUT_DIR / 'contour_puppet_overlay.mp4'
puppet_final = OUTPUT_DIR / 'contour_puppet_only.mp4'
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
overlay_writer = cv2.VideoWriter(str(overlay_raw), fourcc, output_fps, (frame_width, frame_height))
puppet_writer = cv2.VideoWriter(str(puppet_raw), fourcc, output_fps, (frame_width, frame_height))
if not overlay_writer.isOpened() or not puppet_writer.isOpened():
    raise RuntimeError('VideoWriterを開けません。')

state = TrackState()
frame_index = -1
output_frames = 0
rendered_frames = 0
held_frames = 0
track_ids_seen = set()
progress_total = math.ceil(source_frame_count / FRAME_STRIDE) if source_frame_count > 0 else None
if MAX_OUTPUT_FRAMES is not None and progress_total is not None:
    progress_total = min(progress_total, MAX_OUTPUT_FRAMES)

progress = tqdm(total=progress_total, desc='contour puppet render')
while True:
    ok, frame_bgr = capture.read()
    if not ok:
        break
    frame_index += 1
    if frame_index % FRAME_STRIDE != 0:
        continue
    if MAX_OUTPUT_FRAMES is not None and output_frames >= MAX_OUTPUT_FRAMES:
        break

    usable, _ = _infer_person(frame_bgr, state, persist_tracking=True)
    if state.track_id is not None:
        track_ids_seen.add(int(state.track_id))

    if usable:
        overlay_frame, _, _, _ = _render_character(frame_bgr.copy(), state, 'overlay')
        stage_base = np.full_like(frame_bgr, STAGE_BGR)
        ground_y = int(round(0.94 * frame_height))
        cv2.line(
            stage_base,
            (0, ground_y),
            (frame_width - 1, ground_y),
            (180, 185, 190),
            2,
            cv2.LINE_AA,
        )
        puppet_frame, _, _, _ = _render_character(stage_base, state, 'stage')
        rendered_frames += 1
        if state.missing_frames > 0:
            held_frames += 1
    else:
        overlay_frame = frame_bgr
        puppet_frame = np.full_like(frame_bgr, STAGE_BGR)
        cv2.putText(
            puppet_frame,
            'pose / silhouette unavailable',
            (30, 50),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (80, 80, 80),
            2,
            cv2.LINE_AA,
        )

    overlay_writer.write(overlay_frame)
    puppet_writer.write(puppet_frame)
    output_frames += 1
    progress.update(1)

progress.close()
capture.release()
overlay_writer.release()
puppet_writer.release()


def _transcode_with_optional_audio(raw_video, final_video):
    command = [
        'ffmpeg',
        '-y',
        '-loglevel',
        'error',
        '-i',
        str(raw_video),
        '-i',
        str(video_path),
        '-map',
        '0:v:0',
        '-map',
        '1:a:0?',
        '-c:v',
        'libx264',
        '-crf',
        '18',
        '-preset',
        'medium',
        '-pix_fmt',
        'yuv420p',
        '-c:a',
        'aac',
        '-shortest',
        '-movflags',
        '+faststart',
        str(final_video),
    ]
    subprocess.run(command, check=True)


_transcode_with_optional_audio(overlay_raw, overlay_final)
_transcode_with_optional_audio(puppet_raw, puppet_final)

manifest = {
    'schema': 'er_flowscan.contour_puppet.v1',
    'input': {
        'name': video_path.name,
        'bytes': video_path.stat().st_size,
        'sha256': input_sha256,
        'source_fps': source_fps,
        'source_frames': source_frame_count,
    },
    'models': {'pose': POSE_MODEL_NAME, 'segmentation': SEG_MODEL_NAME},
    'render': {
        'mode': PUPPET_MODE,
        'style': PUPPET_STYLE,
        'scale': PUPPET_SCALE,
        'side': PUPPET_SIDE,
        'resolved_side': state.placement_side,
        'pose_smoothing_alpha': POSE_SMOOTHING_ALPHA,
        'bbox_smoothing_alpha': BBOX_SMOOTHING_ALPHA,
        'preserve_part_outlines': PRESERVE_PART_OUTLINES,
        'part_outline_width': PART_OUTLINE_WIDTH,
        'frame_stride': FRAME_STRIDE,
        'output_fps': output_fps,
        'output_frames': output_frames,
        'rendered_frames': rendered_frames,
        'held_frames': held_frames,
        'track_ids_seen': sorted(track_ids_seen),
        'requested_track_id': TARGET_TRACK_ID,
    },
    'outputs': [overlay_final.name, puppet_final.name],
    'limitations': [
        'single-view 2D pose and per-frame person segmentation',
        'not a fixed-identity layered illustration rig',
        'occluded limbs may use a short held state',
    ],
}
manifest_path = OUTPUT_DIR / 'contour_puppet_manifest.json'
manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps(manifest['render'], ensure_ascii=False, indent=2))
print({'overlay': str(overlay_final), 'puppet_only': str(puppet_final), 'manifest': str(manifest_path)})

## 7. 結果表示とダウンロード

ZIPには派生動画とマニフェストだけを入れ、元動画は含めません。

In [ ]:
from IPython.display import Video, display

display(Video(str(overlay_final), embed=True, width=900))
display(Video(str(puppet_final), embed=True, width=900))

download_dir = OUTPUT_DIR / 'download_bundle'
download_dir.mkdir(parents=True, exist_ok=True)
for artifact in (overlay_final, puppet_final, manifest_path):
    shutil.copy2(artifact, download_dir / artifact.name)
archive_path = Path(shutil.make_archive('/content/contour_puppet_results', 'zip', download_dir))
print({'zip': str(archive_path), 'bytes': archive_path.stat().st_size})

DOWNLOAD_RESULTS = True
if DOWNLOAD_RESULTS:
    from google.colab import files

    files.download(str(archive_path))

## 次の改善候補

1. **固定キャラクター化**: 正面基準画像を頭・胴・上腕・前腕・大腿・下腿へ分け、各部位をキーポイント間の相似変換で追従させる。
2. **自然な曲げ**: 三角形メッシュ＋piecewise affine、またはThin Plate Splineへ置き換える。
3. **前後関係**: 深度・手前判定を追加し、腕が胴体の前か後かで描画順を変える。
4. **輪郭の時間安定化**: 光フローでマスクを前フレームから伝播し、境界ジッターを抑える。
5. **顔・手**: MediaPipe Face/Handsを追加し、表情と指を別レイヤーで駆動する。

まずは `PUPPET_MODE='capsule'` と `'hybrid'` を比較し、添付例のような単純な輪郭人形に近い方を選ぶのがおすすめです。